In [ ]:
## No docker
import pandas as pd
pd.set_option('display.max_columns', None)
path = "/home/diego/BigData/financial-fraud-MLOps"

In [10]:
!ls $path/data/raw

cards_data.csv	 mcc_codes.json		    transactions_data.csv
cities15000.txt  train_fraud_labels.json    users_data.csv
countryInfo.txt  train_fraud_labels.ndjson  ZIP_US.txt


In [21]:
transactions_data_dtype = {
    "id": "Int64",

    "client_id": "Int64",
    "card_id": "Int64",
    "amount": "str",   

    "use_chip": "category",
    "merchant_id": "Int64", 
    "merchant_city": "str",
    "merchant_state": "category",
    "zip": "str",
    "mcc": "Int64",
    "errors": "category"                            
}

transactions_data_df = (
    pd.read_csv(
    f"{path}/data/raw/transactions_data.csv", nrows=1000,
    dtype=transactions_data_dtype, parse_dates=["date"], date_format="%Y-%m-%d %H:%M:%S"
    )
    
# Usar este método para más de 11 millones de filas crashea la compu, mejor hacerlo separado
    # .assign(
    #     amount = lambda df: df["amount"].str.replace(r"\$", "", regex=True).astype(float),
    #     zip = lambda df: df["zip"].str.replace(r"\.0$", "", regex=True).str.zfill(5)
    # )
)

transactions_data_df["amount"] = (
    transactions_data_df["amount"]
    .str.replace(r"\$", "", regex=True)
    .astype(float)
)

transactions_data_df["zip"] = (
    transactions_data_df["zip"]
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)            
)

transactions_data_df.info()
transactions_data_df.head() 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              1000 non-null   Int64         
 1   date            1000 non-null   datetime64[ns]
 2   client_id       1000 non-null   Int64         
 3   card_id         1000 non-null   Int64         
 4   amount          1000 non-null   float64       
 5   use_chip        1000 non-null   category      
 6   merchant_id     1000 non-null   Int64         
 7   merchant_city   1000 non-null   object        
 8   merchant_state  865 non-null    category      
 9   zip             864 non-null    object        
 10  mcc             1000 non-null   Int64         
 11  errors          9 non-null      category      
dtypes: Int64(5), category(3), datetime64[ns](1), float64(1), object(2)
memory usage: 81.0+ KB


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,5813,NaN


In [20]:
users_data_dtype = {
    "id": "Int64",
    "current_age": "Int64",
    "retirement_age": "Int64",
    "birth_year": "Int64",
    "birth_month": "Int64",

    "gender": "category",
    "address": str,

    "client_latitude": "Float64",
    "client_longitude": "Float64", 

    "per_capita_income": str,
    "yearly_income": str,
    "total_debt": str,

    "credit_score": "Int64",
    "num_credit_cards": "Int64"                      
}

users_data_df = pd.read_csv(
    f"{path}/data/raw/users_data.csv", nrows=1000, 
    dtype=users_data_dtype
    )

users_data_df = users_data_df.rename(columns={"id": "client_id"})


dollar_columns = ["per_capita_income", "yearly_income", "total_debt"]
users_data_df[dollar_columns] = (
    users_data_df[dollar_columns]
    .stack()
    .str.replace(r"\$", "", regex=True)
    .unstack()
    .astype(float)
)

users_data_df = users_data_df.drop(columns=["address"]) 
users_data_df.info()
users_data_df.head() 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   client_id          1000 non-null   Int64   
 1   current_age        1000 non-null   Int64   
 2   retirement_age     1000 non-null   Int64   
 3   birth_year         1000 non-null   Int64   
 4   birth_month        1000 non-null   Int64   
 5   gender             1000 non-null   category
 6   latitude           1000 non-null   float64 
 7   longitude          1000 non-null   float64 
 8   per_capita_income  1000 non-null   float64 
 9   yearly_income      1000 non-null   float64 
 10  total_debt         1000 non-null   float64 
 11  credit_score       1000 non-null   Int64   
 12  num_credit_cards   1000 non-null   Int64   
dtypes: Int64(7), category(1), float64(5)
memory usage: 101.7 KB


,client_id,current_age,retirement_age,birth_year,birth_month,gender,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,34.15,-117.76,29278.0,59696.0,127613.0,787,5
1,1746,53,68,1966,12,Female,40.76,-73.74,37891.0,77254.0,191349.0,701,5
2,1718,81,67,1938,11,Female,34.02,-117.89,22681.0,33483.0,196.0,698,5
3,708,63,63,1957,1,Female,40.71,-73.99,163145.0,249925.0,202328.0,722,4
4,1164,43,70,1976,9,Male,37.76,-122.44,53797.0,109687.0,183855.0,675,1


In [43]:
cards_data_dtype = {
    "card_id": "Int64",
    "client_id": "Int64",
    "card_brand": "category",
    "card_type": "category",
    "card_number": "str",

    "expires": "str",
    "cvv": "str",
    "has_chip": "str",
    "num_cards_issued": "Int64",

    "credit_limit": "str",

    "acct_open_date": "str", 
    "year_pin_last_changed": "Int64",
    "card_on_dark_web": "str"                 
}

cards_data_df = pd.read_csv(
    f"{path}/data/raw/cards_data.csv", nrows=1000,
    dtype=cards_data_dtype
)

cards_data_df = cards_data_df.rename(columns={"id": "card_id"})

cards_columns_booleans = ["card_on_dark_web", "has_chip"]
cards_booleans = {
    "YES": True,
    "NO": False    
}

cards_data_df[cards_columns_booleans] = (
    cards_data_df[cards_columns_booleans]
    .apply(lambda col: col.str.strip().str.upper().map(cards_booleans).astype("boolean"))
)

cards_data_df["credit_limit"] = (
    cards_data_df["credit_limit"]
    .str.replace(r"\$", "", regex=True)
    .astype(float)
)

cards_data_df["expires"] = pd.to_datetime(
    "01/" + cards_data_df["expires"],
    format = "%d/%m/%Y"
).astype("date32[day][pyarrow]")

cards_data_df["acct_open_date"] = pd.to_datetime(
    "01/" + cards_data_df["acct_open_date"],
    format = "%d/%m/%Y"
).astype("date32[day][pyarrow]")

# print(cards_data_df["has_chip"].unique())
# print(cards_data_df["card_on_dark_web"].unique())
cards_data_df.info()
cards_data_df.head() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype               
---  ------                 --------------  -----               
 0   card_id                1000 non-null   int64               
 1   client_id              1000 non-null   Int64               
 2   card_brand             1000 non-null   category            
 3   card_type              1000 non-null   category            
 4   card_number            1000 non-null   object              
 5   expires                1000 non-null   date32[day][pyarrow]
 6   cvv                    1000 non-null   object              
 7   has_chip               1000 non-null   boolean             
 8   num_cards_issued       1000 non-null   Int64               
 9   credit_limit           1000 non-null   float64             
 10  acct_open_date         1000 non-null   date32[day][pyarrow]
 11  year_pin_last_changed  1000 non-null   Int64

,card_id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,2022-12-01,623,True,2,24295.0,2002-09-01,2008,False
1,2731,825,Visa,Debit,4956965974959986,2020-12-01,393,True,2,21968.0,2014-04-01,2014,False
2,3701,825,Visa,Debit,4582313478255491,2024-02-01,719,True,2,46414.0,2003-07-01,2004,False
3,42,825,Visa,Credit,4879494103069057,2024-08-01,693,False,1,12400.0,2003-01-01,2012,False
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,2009-03-01,75,True,1,28.0,2008-09-01,2009,False


In [26]:
with open(f"{path}/data/raw/mcc_codes.json", "r", encoding="utf-8") as f:
    fragmento = f.read(400)
    print(fragmento)

{
    "5812": "Eating Places and Restaurants",
    "5541": "Service Stations",
    "7996": "Amusement Parks, Carnivals, Circuses",
    "5411": "Grocery Stores, Supermarkets",
    "4784": "Tolls and Bridge Fees",
    "4900": "Utilities - Electric, Gas, Water, Sanitary",
    "5942": "Book Stores",
    "5814": "Fast Food Restaurants",
    "4829": "Money Transfer",
    "5311": "Department Stores",
   


In [35]:
mcc_codes_df = pd.read_json(f"{path}/data/raw/mcc_codes.json", orient="index")

mcc_codes_df = mcc_codes_df.reset_index()
mcc_codes_df.columns = ["mcc", "mcc_description"]

mcc_codes_df.info()
mcc_codes_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   mcc              109 non-null    int64 
 1   mcc_description  109 non-null    object
dtypes: int64(1), object(1)
memory usage: 1.8+ KB


,mcc,mcc_description
0,5812,Eating Places and Restaurants
1,5541,Service Stations
2,7996,"Amusement Parks, Carnivals, Circuses"
3,5411,"Grocery Stores, Supermarkets"
4,4784,Tolls and Bridge Fees


In [17]:
with open(f"{path}/raw/train_fraud_labels.json", "r", encoding="utf-8") as f:
    fragmento = f.read(400)
    print(fragmento)

{"target": {"10649266": "No", "23410063": "No", "9316588": "No", "12478022": "No", "9558530": "No", "12532830": "No", "19526714": "No", "9906964": "No", "13224888": "No", "13749094": "No", "12303776": "No", "19480376": "No", "11716050": "No", "20025400": "No", "7661688": "No", "16662807": "No", "21419778": "No", "18011186": "No", "23289598": "No", "11644547": "No", "23235120": "No", "19748218": "N


In [ ]:
##Ejecutamos el siguiente codigo en la terminal para transformar el json a ndjson
#jq -c '.target | to_entries[] | {id: .key, target: .value}' train_fraud_labels.json > train_fraud_labels.ndjson

In [18]:
with open(f"{path}/raw/train_fraud_labels.ndjson", "r", encoding="utf-8") as f:
    fragmento = f.read(400)
    print(fragmento)

{"id":"10649266","target":"No"}
{"id":"23410063","target":"No"}
{"id":"9316588","target":"No"}
{"id":"12478022","target":"No"}
{"id":"9558530","target":"No"}
{"id":"12532830","target":"No"}
{"id":"19526714","target":"No"}
{"id":"9906964","target":"No"}
{"id":"13224888","target":"No"}
{"id":"13749094","target":"No"}
{"id":"12303776","target":"No"}
{"id":"19480376","target":"No"}
{"id":"11716050","t


In [44]:
labels_dtype = {
    "id": "Int64",
    "target": str,
}

train_fraud_labels_df = pd.read_json(
    f"{path}/data/raw/train_fraud_labels.ndjson", 
    lines=True, nrows=100,
    dtype=labels_dtype
)

labels_booleans = {
    "YES": True,
    "NO": False
}

# print(train_fraud_labels_df["target"].unique())
train_fraud_labels_df["target"] = (
    train_fraud_labels_df["target"]
    .str.strip()
    .str.upper()
    .map(labels_booleans)
    .astype("boolean")
)

train_fraud_labels_df.info()
train_fraud_labels_df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      100 non-null    Int64  
 1   target  100 non-null    boolean
dtypes: Int64(1), boolean(1)
memory usage: 1.2 KB


,id,target
0,10649266,False
1,23410063,False
2,9316588,False
3,12478022,False
4,9558530,False


In [82]:
complete_fraud_data = (
    transactions_data_df
        .merge(mcc_codes_df, on="mcc", how="left")
        .merge(train_fraud_labels_df, on="id", how="left")
        .merge(users_data_df, on="client_id", how="left")
        .merge(cards_data_df, on=["client_id", "card_id"], how="left")
        .drop(columns=["mcc"]) 
)

complete_fraud_data.head(5)

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523,...,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,...,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,...,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,...,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,...,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>


In [85]:
!ls $path/raw

BASEDEDATOS-EDAS2025.xlsx  mcc_codes.json	      transactions_data.csv
cards_data.csv		   train_fraud_labels.json    users_data.csv
json_to_ndjson.txt	   train_fraud_labels.ndjson
